# 📘 Capstone Project: E-Commerce Data Platform
## Build a Complete Production Data Pipeline

**Scenario:** You are the lead data engineer at TechMart. Build a complete data platform
from raw source tables through to business-ready Gold layer analytics.

**Deliverables:**
1. Bronze: 5 source tables ingested with audit trail
2. Silver: Cleaned, validated, deduplicated with quality gates
3. Gold: 4 business analytics tables
4. SCD Type 2 customer dimension
5. Incremental processing (idempotent, safe to re-run)
6. Quality framework + monitoring

**Estimated time:** 6-8 hours

**Prerequisites:** Notebooks 01-20 (techmart_dw must exist)

---

---
# 🏗️ Section 1: Architecture Design

## Pipeline Architecture

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    TECHMART DATA PLATFORM                                 │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│  SOURCE TABLES          BRONZE              SILVER              GOLD     │
│  ─────────────          ──────              ──────              ────     │
│  orders ──────────▶ bronze_orders ────▶ silver_orders ────▶ gold_daily  │
│  customers ───────▶ bronze_customers ─▶ silver_customers ─▶ gold_c360  │
│  products ────────▶ bronze_products ──▶ silver_products ──▶ gold_prod  │
│  payments ────────▶ bronze_payments ──▶ silver_payments       │         │
│  shipments ───────▶ bronze_shipments ─▶ silver_shipments ─▶ gold_ops   │
│                                                                          │
│  QUALITY GATES:    [QG1: Schema]    [QG2: Business]    [QG3: Metrics]  │
│  SCD2:                              dim_customer_scd2                    │
│  MONITORING:       pipeline_metrics, quality_results, pipeline_logs      │
└─────────────────────────────────────────────────────────────────────────┘
```

## Quality Rules per Transition

| Transition | Rule | Threshold |
|---|---|---|
| Source → Bronze | Schema match | 100% |
| Bronze → Silver | PK not null | 100% |
| Bronze → Silver | PK unique | 100% |
| Bronze → Silver | Referential integrity | 99% |
| Bronze → Silver | Completeness (critical cols) | 95% |
| Silver → Gold | Aggregation correctness | 100% |

---
# ⚙️ Section 2: Configuration & Setup

In [0]:
import logging
import json as json_lib
import uuid
import time
from datetime import datetime
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Pipeline configuration
PIPELINE_CONFIG = {
    "name": "techmart_ecommerce_platform",
    "version": "1.0.0",
    "source_db": "techmart_dw",
    "target_db": "techmart_dw",
    "tables": {
        "orders": {"pk": "order_id", "watermark_col": "created_at"},
        "customers": {"pk": "customer_id", "watermark_col": "updated_at"},
        "products": {"pk": "product_id", "watermark_col": "updated_at"},
        "payments": {"pk": "payment_id", "watermark_col": "created_at"},
        "shipments": {"pk": "shipment_id", "watermark_col": "created_at"},
    },
    "quality_thresholds": {
        "completeness": 0.95,
        "uniqueness": 1.0,
        "referential_integrity": 0.99,
    }
}

# Batch metadata
BATCH_ID = f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
CORRELATION_ID = str(uuid.uuid4())[:8]

print(f"Pipeline: {PIPELINE_CONFIG['name']} v{PIPELINE_CONFIG['version']}")
print(f"Batch: {BATCH_ID}")
print(f"Correlation: {CORRELATION_ID}")
print(f"Tables: {list(PIPELINE_CONFIG['tables'].keys())}")

In [0]:
# Utility functions
def add_audit_columns(df, source_table):
    """Add standard audit columns to a DataFrame."""
    return (df
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_system", lit("techmart_oltp"))
        .withColumn("_source_table", lit(source_table))
        .withColumn("_batch_id", lit(BATCH_ID))
        .withColumn("_row_hash", md5(concat_ws("|", *[col(c).cast("string") for c in df.columns])))
    )

def log_step(step_name, status, details=None):
    """Log a pipeline step."""
    entry = {
        "timestamp": datetime.now().isoformat(),
        "batch_id": BATCH_ID,
        "correlation_id": CORRELATION_ID,
        "step": step_name,
        "status": status,
        "details": details or {}
    }
    icon = {"success": "✅", "warning": "⚠️", "error": "❌", "start": "▶️"}.get(status, "ℹ️")
    print(f"  {icon} [{status.upper()}] {step_name}: {details or ''}")
    return entry

print("✅ Utilities ready")

---
# 🥉 Section 3: Bronze Layer

**Rules:** Append-only, no transformations, full audit trail.

In [0]:
# Bronze ingestion for all source tables
spark.sql("USE techmart_dw")

def ingest_to_bronze(table_name, config):
    """Ingest a source table into Bronze with audit columns."""
    log_step(f"bronze_{table_name}", "start")
    start = time.time()
    
    source_df = spark.table(f"{config['source_db']}.{table_name}")
    bronze_df = add_audit_columns(source_df, table_name)
    
    target = f"capstone_bronze_{table_name}"
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(target)
    
    count = spark.table(target).count()
    duration = time.time() - start
    log_step(f"bronze_{table_name}", "success", {"rows": count, "duration_s": round(duration, 2)})
    return count

# Ingest all tables
print("=" * 60)
print("BRONZE LAYER INGESTION")
print("=" * 60)
bronze_counts = {}
for table in PIPELINE_CONFIG["tables"]:
    bronze_counts[table] = ingest_to_bronze(table, PIPELINE_CONFIG)

print(f"\nBronze totals: {sum(bronze_counts.values()):,} rows across {len(bronze_counts)} tables")

In [0]:
# ============================================
# 🎯 YOUR TURN: Bronze for website_events
# ============================================
# Ingest website_events (100K rows) into Bronze:
# 1. Read from techmart_dw.website_events
# 2. Add audit columns
# 3. Write to capstone_bronze_website_events
# 4. Log the result
# Hint: This is a high-volume table — note the timing
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
start = time.time()
events = spark.table("website_events")
bronze_events = add_audit_columns(events, "website_events")
bronze_events.write.format("delta").mode("overwrite").saveAsTable("capstone_bronze_website_events")
count = spark.table("capstone_bronze_website_events").count()
print(f"✅ Bronze website_events: {count:,} rows ({time.time()-start:.1f}s)")

---
# 🥈 Section 4: Silver Layer

**Rules:** Deduplicated, cleaned, validated, quality-gated.

In [0]:
# Silver processing with quality gates
def process_silver(bronze_table, pk_col, quality_rules=None):
    """Process Bronze → Silver with dedup, cleaning, and quality gate."""
    log_step(f"silver_{bronze_table}", "start")
    
    df = spark.table(f"capstone_{bronze_table}")
    initial_count = df.count()
    
    # Step 1: Deduplicate (keep latest by _ingested_at)
    w = Window.partitionBy(pk_col).orderBy(col("_ingested_at").desc())
    deduped = df.withColumn("_rn", row_number().over(w)).filter("_rn = 1").drop("_rn")
    
    # Step 2: Clean (standardize strings, handle nulls)
    for c in deduped.columns:
        if c.startswith("_"):
            continue
        dtype = str(deduped.schema[c].dataType)
        if "String" in dtype:
            deduped = deduped.withColumn(c, trim(col(c)))
    
    # Step 3: Validate (remove null PKs)
    validated = deduped.filter(f"{pk_col} IS NOT NULL")
    
    # Step 4: Add processing metadata
    silver = validated.withColumn("_processed_at", current_timestamp())
    
    # Write
    target = bronze_table.replace("bronze_", "silver_")
    silver.write.format("delta").mode("overwrite").saveAsTable(f"capstone_{target}")
    
    final_count = spark.table(f"capstone_{target}").count()
    dropped = initial_count - final_count
    
    log_step(f"silver_{bronze_table}", "success", {
        "input": initial_count, "output": final_count, "dropped": dropped
    })
    return final_count

# Process all tables
print("=" * 60)
print("SILVER LAYER PROCESSING")
print("=" * 60)
silver_counts = {}
for table, config in PIPELINE_CONFIG["tables"].items():
    silver_counts[table] = process_silver(f"bronze_{table}", config["pk"])

print(f"\nSilver totals: {sum(silver_counts.values()):,} rows")

In [0]:
# ============================================
# 🎯 YOUR TURN: Silver with Quality Gate
# ============================================
# Build a quality gate function that:
# 1. Checks PK uniqueness (must be 100%)
# 2. Checks completeness of critical columns (> 95%)
# 3. Returns pass/fail with metrics
# 4. Apply to capstone_silver_orders
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
def quality_gate(table_name, pk_col, critical_cols):
    """Run quality gate on a Silver table."""
    df = spark.table(table_name)
    total = df.count()
    results = []
    
    # PK uniqueness
    distinct_pks = df.select(pk_col).distinct().count()
    pk_unique = distinct_pks == total
    results.append({"check": "pk_unique", "passed": pk_unique, "value": f"{distinct_pks}/{total}"})
    
    # Completeness
    for col_name in critical_cols:
        non_null = df.filter(f"{col_name} IS NOT NULL").count()
        completeness = non_null / total
        passed = completeness >= 0.95
        results.append({"check": f"completeness_{col_name}", "passed": passed, "value": f"{completeness:.3f}"})
    
    # Report
    all_passed = all(r["passed"] for r in results)
    icon = "✅" if all_passed else "❌"
    print(f"{icon} Quality Gate: {table_name}")
    for r in results:
        status = "✅" if r["passed"] else "❌"
        print(f"    {status} {r['check']}: {r['value']}")
    return all_passed

quality_gate("capstone_silver_orders", "order_id", ["customer_id", "total_amount", "status"])

---
# 🥇 Section 5: Gold Layer

Build 4 business analytics tables.

In [0]:
# Gold 1: Daily Sales Dashboard
from pyspark.sql.functions import *

orders = spark.table("capstone_silver_orders")

gold_daily = (orders
    .filter("status = 'completed'")
    .withColumn("order_year", year("order_date"))
    .withColumn("order_month", month("order_date"))
    .groupBy("order_date", "payment_method", "order_source")
    .agg(
        count("*").alias("order_count"),
        countDistinct("customer_id").alias("unique_customers"),
        round(sum("total_amount"), 2).alias("revenue"),
        round(avg("total_amount"), 2).alias("avg_order_value")
    )
    .withColumn("_computed_at", current_timestamp())
)
gold_daily.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_daily_sales")
print(f"✅ Gold daily sales: {spark.table('capstone_gold_daily_sales').count():,} rows")

In [0]:
# Gold 2: Customer 360
orders = spark.table("capstone_silver_orders")
customers = spark.table("capstone_silver_customers")

order_metrics = (orders
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_orders"),
        sum(when(col("status")=="completed",1).otherwise(0)).alias("completed_orders"),
        round(sum("total_amount"), 2).alias("lifetime_value"),
        round(avg("total_amount"), 2).alias("avg_order_value"),
        min("order_date").alias("first_order"),
        max("order_date").alias("last_order"),
        countDistinct("payment_method").alias("payment_methods_used")
    )
    .withColumn("days_since_last", datediff(current_date(), col("last_order")))
    .withColumn("churn_risk",
        when(col("days_since_last") > 365, "High")
        .when(col("days_since_last") > 180, "Medium")
        .otherwise("Low"))
)

gold_c360 = (customers
    .select("customer_id", "first_name", "last_name", "customer_segment", "city", "state")
    .join(order_metrics, "customer_id", "left")
    .withColumn("_computed_at", current_timestamp())
)
gold_c360.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_customer_360")
print(f"✅ Gold customer 360: {spark.table('capstone_gold_customer_360').count():,} rows")

In [0]:
# Gold 3: Product Performance
orders = spark.table("capstone_silver_orders")
items = spark.table("techmart_dw.order_items")
products = spark.table("capstone_silver_products")

gold_product = (items
    .join(orders.select("order_id", "status"), "order_id")
    .join(products.select("product_id", "product_name", "category", "brand", "price", "cost"), "product_id")
    .groupBy("product_id", "product_name", "category", "brand")
    .agg(
        count("*").alias("units_sold"),
        round(sum("line_total"), 2).alias("revenue"),
        round(avg("unit_price"), 2).alias("avg_selling_price"),
        sum(when(col("status")=="returned", col("quantity")).otherwise(0)).alias("units_returned")
    )
    .withColumn("return_rate", round(col("units_returned") / col("units_sold") * 100, 2))
    .withColumn("_computed_at", current_timestamp())
)
gold_product.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_product_performance")
print(f"✅ Gold product performance: {spark.table('capstone_gold_product_performance').count():,} rows")

In [0]:
# ============================================
# 🎯 YOUR TURN: Gold 4 — Operational Health
# ============================================
# Build capstone_gold_operational_health with:
# 1. Table freshness: max(_ingested_at) per bronze table
# 2. Row counts per layer (bronze, silver, gold)
# 3. Quality gate pass rates
# Hint: Query the capstone_bronze_* and capstone_silver_* tables
#
# Write your code below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
import pandas as pd
from datetime import datetime

metrics = []
for table in ["orders", "customers", "products", "payments", "shipments"]:
    for layer in ["bronze", "silver"]:
        t = f"capstone_{layer}_{table}"
        try:
            count = spark.table(t).count()
            metrics.append({"table": t, "layer": layer, "row_count": count, "checked_at": datetime.now().isoformat()})
        except:
            pass

metrics_df = spark.createDataFrame(pd.DataFrame(metrics))
metrics_df.write.format("delta").mode("overwrite").saveAsTable("capstone_gold_operational_health")
print(f"✅ Gold operational health: {metrics_df.count()} metrics")
metrics_df.show(truncate=False)

---
# 📐 Section 6: SCD Type 2 Customer Dimension

In [0]:
%sql
-- SCD2 Customer Dimension
USE techmart_dw;
DROP TABLE IF EXISTS capstone_dim_customer_scd2;
CREATE TABLE capstone_dim_customer_scd2 (
  customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  customer_id INT, first_name STRING, last_name STRING,
  city STRING, state STRING, customer_segment STRING,
  effective_start DATE, effective_end DATE, is_current BOOLEAN,
  _row_hash STRING
);

-- Initial load
INSERT INTO capstone_dim_customer_scd2 (customer_id, first_name, last_name, city, state, customer_segment, effective_start, effective_end, is_current, _row_hash)
SELECT customer_id, first_name, last_name, city, state, customer_segment,
  CAST(registration_date AS DATE), CAST('9999-12-31' AS DATE), true,
  MD5(CONCAT(COALESCE(first_name,''), COALESCE(city,''), COALESCE(state,''), COALESCE(customer_segment,'')))
FROM capstone_silver_customers;

SELECT COUNT(*) AS scd2_rows, SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows
FROM capstone_dim_customer_scd2;

---
# 🔍 Section 7: Quality Framework

In [0]:
# Quality results tracking
import pandas as pd
from datetime import datetime

def run_quality_suite(tables_config):
    """Run quality checks across all tables and store results."""
    results = []
    
    for table, checks in tables_config.items():
        df = spark.table(table)
        total = df.count()
        
        for check_name, check_expr in checks.items():
            passing = df.filter(check_expr).count()
            rate = passing / total if total > 0 else 0
            results.append({
                "table": table, "check": check_name,
                "total_rows": total, "passing_rows": passing,
                "pass_rate": round(rate, 4),
                "status": "PASS" if rate >= 0.95 else "WARN" if rate >= 0.90 else "FAIL",
                "checked_at": datetime.now().isoformat()
            })
    
    results_df = spark.createDataFrame(pd.DataFrame(results))
    results_df.write.format("delta").mode("append").saveAsTable("capstone_quality_results")
    
    # Report
    print(f"Quality Suite Results:")
    for r in results:
        icon = {"PASS": "✅", "WARN": "⚠️", "FAIL": "❌"}[r["status"]]
        print(f"  {icon} {r['table'].split('_')[-1]}.{r['check']}: {r['pass_rate']:.1%}")
    
    return results

# Run quality checks
quality_config = {
    "capstone_silver_orders": {
        "pk_not_null": "order_id IS NOT NULL",
        "amount_positive": "total_amount >= 0",
        "valid_status": "status IN ('completed','shipped','processing','pending','cancelled','returned')",
    },
    "capstone_silver_customers": {
        "pk_not_null": "customer_id IS NOT NULL",
        "has_name": "first_name IS NOT NULL",
    }
}

run_quality_suite(quality_config)

---
# 🔄 Section 8: Orchestration

In [0]:
# Pipeline orchestration with dependency tracking
import time

class PipelineOrchestrator:
    """Orchestrate pipeline stages with dependency management."""
    
    def __init__(self, name):
        self.name = name
        self.completed = set()
        self.metrics = []
    
    def run_stage(self, stage_name, func, depends_on=None):
        """Run a stage if dependencies are met."""
        deps = depends_on or []
        unmet = [d for d in deps if d not in self.completed]
        if unmet:
            print(f"  ⏭️ Skipping {stage_name} (unmet deps: {unmet})")
            return False
        
        start = time.time()
        try:
            result = func()
            self.completed.add(stage_name)
            duration = time.time() - start
            self.metrics.append({"stage": stage_name, "status": "success", "duration_s": round(duration, 2)})
            print(f"  ✅ {stage_name} ({duration:.1f}s)")
            return True
        except Exception as e:
            self.metrics.append({"stage": stage_name, "status": "failed", "error": str(e)})
            print(f"  ❌ {stage_name}: {e}")
            return False
    
    def summary(self):
        total_time = sum(m["duration_s"] for m in self.metrics if "duration_s" in m)
        print(f"\n{'='*50}")
        print(f"Pipeline: {self.name}")
        print(f"Stages: {len(self.completed)}/{len(self.metrics)} completed")
        print(f"Total time: {total_time:.1f}s")
        print(f"{'='*50}")

# Demo orchestration
orch = PipelineOrchestrator("techmart_capstone")
orch.run_stage("verify_source", lambda: spark.table("techmart_dw.orders").count())
orch.run_stage("bronze", lambda: "done", depends_on=["verify_source"])
orch.run_stage("silver", lambda: "done", depends_on=["bronze"])
orch.run_stage("gold", lambda: "done", depends_on=["silver"])
orch.summary()

---
# 📋 Section 9: Final Verification

In [0]:
%sql
-- Final table catalog
SELECT 'capstone_bronze_orders' AS table_name, COUNT(*) AS rows FROM capstone_bronze_orders
UNION ALL SELECT 'capstone_bronze_customers', COUNT(*) FROM capstone_bronze_customers
UNION ALL SELECT 'capstone_silver_orders', COUNT(*) FROM capstone_silver_orders
UNION ALL SELECT 'capstone_silver_customers', COUNT(*) FROM capstone_silver_customers
UNION ALL SELECT 'capstone_gold_daily_sales', COUNT(*) FROM capstone_gold_daily_sales
UNION ALL SELECT 'capstone_gold_customer_360', COUNT(*) FROM capstone_gold_customer_360
UNION ALL SELECT 'capstone_gold_product_performance', COUNT(*) FROM capstone_gold_product_performance
UNION ALL SELECT 'capstone_dim_customer_scd2', COUNT(*) FROM capstone_dim_customer_scd2
ORDER BY table_name;

---
# 📗 Enhancement: E-Commerce Pipeline -- Production Patterns

## Architecture Overview

```
Source Systems
    |
    v
Bronze Layer (raw, append-only)
    |
    v
Silver Layer (cleaned, validated, deduplicated)
    |
    v
Gold Layer (aggregated, business-ready)
    |
    v
Serving (SQL Warehouse, APIs, ML)
```

## Key Tables: orders, customers, products, order_items

## Production Checklist

| Item | Status |
|------|--------|
| Idempotent pipeline (safe to rerun) | Required |
| Data quality checks at each layer | Required |
| Incremental processing (not full refresh) | Required |
| Error handling and dead letter queue | Required |
| Monitoring and alerting | Required |
| Documentation and data catalog | Required |

## Common Patterns Used

1. **Auto Loader** for file ingestion
2. **MERGE** for CDC and upserts
3. **ROW_NUMBER** for deduplication
4. **Window functions** for running totals, rankings
5. **DLT expectations** for quality gates
6. **Lakeflow Jobs** for orchestration

In [0]:
# E-Commerce Pipeline -- Key SQL Patterns
print("E-Commerce Pipeline -- Production SQL Patterns")
print("=" * 60)

# Pattern 1: Incremental load with watermark
print("\n1. Incremental load pattern:")
print("""
    MERGE INTO silver.orders AS target
    USING (
        SELECT * FROM bronze.raw_orders
        WHERE _ingested_at > (SELECT MAX(_ingested_at) FROM silver.orders)
    ) AS source
    ON target.id = source.id
    WHEN MATCHED AND source._row_hash != target._row_hash THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

# Pattern 2: Deduplication
print("2. Deduplication pattern:")
print("""
    SELECT * FROM (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY _ingested_at DESC
            ) AS rn
        FROM bronze.raw_orders
    ) WHERE rn = 1
""")

# Pattern 3: Quality checks
print("3. Quality check pattern:")
quality_checks = [
    f"SELECT COUNT(*) FROM silver.orders WHERE id IS NULL",
    f"SELECT COUNT(*) - COUNT(DISTINCT id) FROM silver.orders",
    f"SELECT COUNT(*) FROM silver.orders WHERE amount < 0",
]
for qc in quality_checks:
    print(f"   {qc}")


---
# ✅ Section 10: Review Checklist

- [x] All Bronze tables have audit columns (_ingested_at, _source_system, _batch_id, _row_hash)
- [x] All Silver tables are deduplicated (ROW_NUMBER on PK)
- [x] All Gold tables have business value (daily sales, customer 360, product performance)
- [x] SCD2 supports point-in-time queries (effective_start/end, is_current)
- [x] Pipeline is idempotent (overwrite mode, deterministic transforms)
- [x] Quality gates are enforced (PK uniqueness, completeness, valid values)
- [x] Monitoring is in place (operational health table, quality results)
- [x] Documentation is complete (architecture diagram, table catalog)

## What Would You Improve?

1. **Streaming:** Replace batch with Structured Streaming for real-time
2. **Alerting:** Integrate with Slack/PagerDuty for quality failures
3. **Lineage:** Add column-level lineage tracking
4. **Testing:** Add property-based tests for edge cases
5. **Partitioning:** Add Liquid Clustering to Gold tables
6. **Security:** Add column masking for PII in Silver

---
# 🎓 Capstone Complete!

**What was built:**
- 5 Bronze tables with full audit trail
- 5 Silver tables with dedup, cleaning, validation
- 4 Gold analytics tables (daily sales, customer 360, product performance, operational health)
- SCD Type 2 customer dimension
- Quality framework with results tracking
- Pipeline orchestrator with dependency management

**Next:** Notebook 22 — Capstone: FinTech Pipeline